# Computer Vision — Assignment 2
## Step-by-Step Google Colab Notebook

**Name:** Kishan Arya  , Sidharth 
**Course:** Computer Vision  
**Assignment:** Assignment 2

This notebook follows the four questions in the assignment. Every major operation is separated into cells so that the intermediate results can be inspected and explained during evaluation.

### Questions covered
1. Line Structure Detection using **LoG + zero crossing + Hough Transform**
2. Robust Circle Detection using **Hough Transform**
3. Effect of **Gaussian noise on thresholding**
4. **Otsu's thresholding implemented from scratch**

> **Important:** The notebook uses standard `scikit-image` sample images so it runs directly in Colab. You can replace them with your own images later if required.

## 0. Install / import required libraries

We use:
- `NumPy` for numerical operations
- `Matplotlib` for visualization
- `SciPy` for convolution
- `scikit-image` for image data, Canny edges and Hough transforms

The LoG zero-crossing detector and Otsu algorithm are implemented explicitly rather than hiding the main logic inside a single built-in function.


In [ ]:
# Run this cell first in Google Colab
!pip -q install scikit-image scipy matplotlib pandas


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from scipy import ndimage
from skimage import data, color, img_as_float
from skimage.feature import canny
from skimage.transform import hough_line, hough_line_peaks
from skimage.transform import hough_circle, hough_circle_peaks
from skimage.draw import circle_perimeter
from skimage.filters import gaussian

plt.rcParams["figure.figsize"] = (9, 6)
plt.rcParams["image.cmap"] = "gray"

print("Libraries imported successfully.")

# Q1. Line Structure Detection using LoG and Hough Transform

The assignment asks us to:
1. Convert a line-rich image to grayscale.
2. Apply the **Laplacian of Gaussian (LoG)**.
3. Obtain edges using **zero-crossing detection**.
4. Repeat with two LoG thresholds.
5. Apply the Hough Transform.
6. Overlay detected lines and compare the two thresholds.

We use `data.page()` because it contains many strong approximately linear structures (text rows and page boundaries), making it useful for demonstrating line detection.


In [ ]:
# Load a standard grayscale image containing many linear structures
img_q1 = img_as_float(data.page())

print("Image shape:", img_q1.shape)
print("Intensity range:", (img_q1.min(), img_q1.max()))

plt.figure(figsize=(8, 6))
plt.imshow(img_q1, cmap="gray")
plt.title("Q1 — Original Grayscale Image")
plt.axis("off")
plt.show()


### Step 1 — Smooth the image before taking the Laplacian

The Laplacian is a second-order derivative and is sensitive to noise. Therefore, we first smooth the image with a Gaussian filter.

Then:

\[
LoG(I) = \nabla^2(G_\sigma * I)
\]

where \(G_\sigma\) is a Gaussian kernel.


In [ ]:
sigma_q1 = 2.0

smoothed_q1 = gaussian(img_q1, sigma=sigma_q1)
log_response = ndimage.laplace(smoothed_q1)

fig, ax = plt.subplots(1, 3, figsize=(15, 5))
ax[0].imshow(img_q1, cmap="gray")
ax[0].set_title("Original")
ax[1].imshow(smoothed_q1, cmap="gray")
ax[1].set_title(f"Gaussian Smoothed (σ={sigma_q1})")
ax[2].imshow(log_response, cmap="gray")
ax[2].set_title("Laplacian of Gaussian Response")
for a in ax:
    a.axis("off")
plt.tight_layout()
plt.show()

print("LoG response range:", log_response.min(), "to", log_response.max())


### Step 2 — Zero-crossing edge detector

An edge occurs near a point where the LoG response changes sign.

For every pixel we inspect neighboring pixels. If there is a positive/negative sign change, the pixel is considered a zero-crossing candidate.

However, very small sign changes can be caused by weak/noisy responses. Therefore we introduce a threshold on the **magnitude of the LoG response around the crossing**.

We deliberately implement this step ourselves.


In [ ]:
def zero_crossing_log(log_img, threshold=0.01):
    """Zero-crossing detector with a response-magnitude threshold."""
    log_img = np.asarray(log_img, dtype=float)

    # Neighbor pairs: horizontal, vertical, and the two diagonals
    shifts = [(0, 1), (1, 0), (1, 1), (1, -1)]
    edge = np.zeros(log_img.shape, dtype=bool)

    for dr, dc in shifts:
        shifted = np.roll(log_img, (dr, dc), axis=(0, 1))

        # Sign change, excluding exact zeros
        sign_change = ((log_img > 0) & (shifted < 0)) | ((log_img < 0) & (shifted > 0))

        # Keep only meaningful responses
        strong = np.maximum(np.abs(log_img), np.abs(shifted)) >= threshold
        edge |= sign_change & strong

    # Remove wrap-around artifacts introduced by np.roll
    edge[0, :] = False
    edge[-1, :] = False
    edge[:, 0] = False
    edge[:, -1] = False

    return edge


threshold_low = 0.005
threshold_high = 0.02

edges_low = zero_crossing_log(log_response, threshold_low)
edges_high = zero_crossing_log(log_response, threshold_high)

print("Low threshold :", threshold_low, " | edge pixels =", edges_low.sum())
print("High threshold:", threshold_high, " | edge pixels =", edges_high.sum())


In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 5))

ax[0].imshow(img_q1, cmap="gray")
ax[0].set_title("Original")

ax[1].imshow(edges_low, cmap="gray")
ax[1].set_title(f"LoG Zero Crossing — Low Threshold = {threshold_low}")

ax[2].imshow(edges_high, cmap="gray")
ax[2].set_title(f"LoG Zero Crossing — High Threshold = {threshold_high}")

for a in ax:
    a.axis("off")
plt.tight_layout()
plt.show()


### Step 3 — Hough Transform for line detection

For a line,

\[
x\cos\theta + y\sin\theta = \rho
\]

The Hough Transform maps edge pixels into \((\rho,\theta)\) parameter space. Peaks in this accumulator correspond to likely lines.

The assignment allows a built-in Hough Transform, so `hough_line` and `hough_line_peaks` are used here.


In [ ]:
def detect_hough_lines(edge_image, num_peaks=12, threshold_fraction=0.25):
    hspace, angles, distances = hough_line(edge_image)

    peak_threshold = threshold_fraction * hspace.max()
    accums, peak_angles, peak_distances = hough_line_peaks(
        hspace, angles, distances,
        threshold=peak_threshold,
        num_peaks=num_peaks
    )

    return hspace, angles, distances, accums, peak_angles, peak_distances


results_low = detect_hough_lines(edges_low, num_peaks=15, threshold_fraction=0.25)
results_high = detect_hough_lines(edges_high, num_peaks=15, threshold_fraction=0.25)

print("Detected lines with LOW LoG threshold :", len(results_low[3]))
print("Detected lines with HIGH LoG threshold:", len(results_high[3]))


In [ ]:
def overlay_hough_lines(image, peak_angles, peak_distances, title):
    fig, ax = plt.subplots(figsize=(10, 7))
    ax.imshow(image, cmap="gray")

    rows, cols = image.shape

    for angle, dist in zip(peak_angles, peak_distances):
        # x*cos(theta) + y*sin(theta) = rho
        if np.abs(np.sin(angle)) > 1e-10:
            x0, x1 = 0, cols - 1
            y0 = (dist - x0 * np.cos(angle)) / np.sin(angle)
            y1 = (dist - x1 * np.cos(angle)) / np.sin(angle)
            ax.plot([x0, x1], [y0, y1], linewidth=1.5)

    ax.set_xlim(0, cols)
    ax.set_ylim(rows, 0)
    ax.set_title(title)
    ax.axis("off")
    plt.show()


overlay_hough_lines(
    img_q1, results_low[4], results_low[5],
    f"Lines from LOW LoG Threshold ({threshold_low})"
)

overlay_hough_lines(
    img_q1, results_high[4], results_high[5],
    f"Lines from HIGH LoG Threshold ({threshold_high})"
)


In [ ]:
# Show the Hough accumulator for the two LoG settings
fig, ax = plt.subplots(1, 2, figsize=(15, 5))

ax[0].imshow(
    results_low[0],
    extent=[np.rad2deg(results_low[1][0]), np.rad2deg(results_low[1][-1]),
            results_low[2][-1], results_low[2][0]],
    aspect="auto"
)
ax[0].set_title("Hough Accumulator — Low LoG Threshold")
ax[0].set_xlabel("θ (degrees)")
ax[0].set_ylabel("ρ")

ax[1].imshow(
    results_high[0],
    extent=[np.rad2deg(results_high[1][0]), np.rad2deg(results_high[1][-1]),
            results_high[2][-1], results_high[2][0]],
    aspect="auto"
)
ax[1].set_title("Hough Accumulator — High LoG Threshold")
ax[1].set_xlabel("θ (degrees)")
ax[1].set_ylabel("ρ")

plt.tight_layout()
plt.show()


### Q1 — Discussion

**Effect of the LoG threshold:**
- A **low threshold** keeps more zero crossings. This can preserve weak line structures, but it also increases false edge responses.
- A **high threshold** suppresses weak responses. The edge map becomes cleaner, but weak or faint lines may disappear.
- Since the Hough Transform works on the resulting edge map, changing the LoG threshold changes the number and strength of Hough peaks.
- The best threshold is therefore a compromise between **line preservation** and **false detections**.

For the report, record the actual line counts printed by the notebook and use the two overlay figures as evidence.


# Q2. Robust Circle Detection using Hough Transform

The assignment requires circle detection on an image containing multiple circular objects with different sizes.

We use `skimage.data.coins()`, which contains several coin-like circular objects. The workflow is:

**Original → grayscale → Canny edges → circular Hough transform → select peaks → overlay circles**

We will then vary:
- radius range
- detection threshold


In [ ]:
coins = img_as_float(data.coins())

plt.figure(figsize=(9, 6))
plt.imshow(coins, cmap="gray")
plt.title("Q2 — Original Coins Image")
plt.axis("off")
plt.show()

print("Image shape:", coins.shape)


In [ ]:
# Step 1 — Edge detection
edges_coins = canny(coins, sigma=2, low_threshold=0.1, high_threshold=0.25)

plt.figure(figsize=(9, 6))
plt.imshow(edges_coins, cmap="gray")
plt.title("Canny Edge Map Used for Circle Hough Transform")
plt.axis("off")
plt.show()


### Step 2 — Define a radius range

The radius range should roughly cover the sizes of the circular objects.

A range that is too narrow can miss coins.
A range that is too broad increases computation and can introduce false circles.

We first use a broad range and then tune it.


In [ ]:
# Candidate radii in pixels
radii = np.arange(15, 55, 2)

hough_radii = hough_circle(edges_coins, radii)

accums, cx, cy, detected_radii = hough_circle_peaks(
    hough_radii,
    radii,
    total_num_peaks=20,
    threshold=0.45
)

print("Number of detected circles:", len(detected_radii))
print("Detected radii:", detected_radii)


In [ ]:
def draw_detected_circles(image, cx, cy, radii, title):
    fig, ax = plt.subplots(figsize=(10, 7))
    ax.imshow(image, cmap="gray")

    for center_x, center_y, radius in zip(cx, cy, radii):
        circy, circx = circle_perimeter(
            int(center_y), int(center_x), int(radius),
            shape=image.shape
        )
        ax.plot(circx, circy, linewidth=2)

    ax.set_title(title)
    ax.axis("off")
    plt.show()


draw_detected_circles(
    coins, cx, cy, detected_radii,
    "Detected Circles — Initial Parameters"
)


### Step 3 — Investigate detection threshold

The Hough accumulator measures how strongly the image supports a particular circle.

- Lower threshold → more candidate circles, including false positives.
- Higher threshold → fewer, stronger circles, but some true circles may be missed.

We compare several thresholds.


In [ ]:
thresholds = [0.30, 0.45, 0.60]

circle_threshold_results = []

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, th in zip(axes, thresholds):
    acc, x, y, r = hough_circle_peaks(
        hough_radii, radii,
        total_num_peaks=20,
        threshold=th
    )

    circle_threshold_results.append((th, len(r)))

    ax.imshow(coins, cmap="gray")
    for cx_i, cy_i, r_i in zip(x, y, r):
        circy, circx = circle_perimeter(
            int(cy_i), int(cx_i), int(r_i), shape=coins.shape
        )
        ax.plot(circx, circy, linewidth=1.5)

    ax.set_title(f"Threshold = {th}\n{len(r)} circles")
    ax.axis("off")

plt.tight_layout()
plt.show()

pd.DataFrame(circle_threshold_results, columns=["Hough threshold", "Detected circles"])


### Step 4 — Investigate radius range

Now we keep the threshold fixed and compare different radius ranges.

This demonstrates why the radius range is an important Hough parameter.


In [ ]:
radius_ranges = [
    np.arange(15, 40, 2),
    np.arange(15, 55, 2),
    np.arange(25, 65, 2)
]

range_results = []

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, rr in zip(axes, radius_ranges):
    hc = hough_circle(edges_coins, rr)
    acc, x, y, r = hough_circle_peaks(
        hc, rr,
        total_num_peaks=20,
        threshold=0.45
    )

    range_results.append([
        f"{rr.min()}–{rr.max()}",
        len(r)
    ])

    ax.imshow(coins, cmap="gray")
    for cx_i, cy_i, r_i in zip(x, y, r):
        circy, circx = circle_perimeter(
            int(cy_i), int(cx_i), int(r_i), shape=coins.shape
        )
        ax.plot(circx, circy, linewidth=1.5)

    ax.set_title(f"Radius = {rr.min()}–{rr.max()}\n{len(r)} circles")
    ax.axis("off")

plt.tight_layout()
plt.show()

pd.DataFrame(range_results, columns=["Radius range", "Detected circles"])


### Q2 — Parameter conclusion

A good parameter setting should:
- detect most visible circular objects,
- place circles close to their actual boundaries,
- avoid many duplicate/false circles.

For this image, use the comparison figures and tables above to select the final radius range and Hough threshold. In the final report, explain that **missed detections** can occur when boundaries are incomplete, weak, noisy, or outside the selected radius range, while **false detections** can occur when unrelated edge fragments accidentally form strong circular patterns.


# Q3. The Role of Noise in Image Thresholding

The assignment asks us to:
1. Start with a clean grayscale image.
2. Add Gaussian noise with mean 0 and \(\sigma=10\) and \(\sigma=50\).
3. Display noisy images and their intensity histograms.
4. Apply a global threshold.
5. Discuss how increasing noise affects histogram separation and segmentation.

We use a grayscale image and keep the same global threshold for a direct comparison.


In [ ]:
# load the clean image 
img_q3 = img_as_float(data.camera())
rng = np.random.default_rng(42)

def add_gaussian_noise(image, sigma_255, rng):
    # sigma_255 is specified in 8-bit intensity units
    sigma = sigma_255 / 255.0
    noisy = image + rng.normal(0, sigma, image.shape)
    return np.clip(noisy, 0, 1)

noisy_10 = add_gaussian_noise(img_q3, 10, rng)
noisy_50 = add_gaussian_noise(img_q3, 50, rng)

fig, ax = plt.subplots(1, 3, figsize=(17, 5))
ax[0].imshow(img_q3, cmap="gray", vmin=0, vmax=1)
ax[0].set_title("Clean Image")
ax[1].imshow(noisy_10, cmap="gray", vmin=0, vmax=1)
ax[1].set_title("Gaussian Noise σ = 10")
ax[2].imshow(noisy_50, cmap="gray", vmin=0, vmax=1)
ax[2].set_title("Gaussian Noise σ = 50")
for a in ax:
    a.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
# Compare intensity histograms
fig, ax = plt.subplots(1, 3, figsize=(18, 5))

ax[0].hist(img_q3.ravel(), bins=256, range=(0, 1), density=True)
ax[0].set_title("Histogram — Clean")
ax[0].set_xlabel("Normalized intensity")
ax[0].set_ylabel("Probability density")

ax[1].hist(noisy_10.ravel(), bins=256, range=(0, 1), density=True)
ax[1].set_title("Histogram — Gaussian σ = 10")
ax[1].set_xlabel("Normalized intensity")

ax[2].hist(noisy_50.ravel(), bins=256, range=(0, 1), density=True)
ax[2].set_title("Histogram — Gaussian σ = 50")
ax[2].set_xlabel("Normalized intensity")

plt.tight_layout()
plt.show()


### Step 3 — Apply a fixed global threshold

To isolate the effect of noise, we use the **same threshold** for all three images.

For demonstration:

\[
T = 0.50
\]

Pixels with intensity \(\geq T\) become foreground (1); the rest become background (0).


In [ ]:
T_global = 0.50

binary_clean = img_q3 >= T_global
binary_10 = noisy_10 >= T_global
binary_50 = noisy_50 >= T_global

fig, ax = plt.subplots(1, 3, figsize=(17, 5))

ax[0].imshow(binary_clean, cmap="gray")
ax[0].set_title(f"Clean — T = {T_global}")

ax[1].imshow(binary_10, cmap="gray")
ax[1].set_title(f"σ = 10 — T = {T_global}")

ax[2].imshow(binary_50, cmap="gray")
ax[2].set_title(f"σ = 50 — T = {T_global}")
for a in ax:
    a.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
# Quantify how much the binary result changes relative to the clean thresholded image
def binary_change(reference, test):
    return np.mean(reference != test) * 100

print(f"Changed pixels for σ=10: {binary_change(binary_clean, binary_10):.2f}%")
print(f"Changed pixels for σ=50: {binary_change(binary_clean, binary_50):.2f}%")


### Q3 — Discussion

As Gaussian noise increases from \(\sigma=10\) to \(\sigma=50\):

1. Histogram peaks become **broader**.
2. Intensities from different regions increasingly **overlap**.
3. A single threshold becomes less reliable.
4. The binary image contains more false foreground/background pixels.
5. Fine structures are more likely to be lost or fragmented.

Thus, higher noise makes global threshold selection harder because the histogram no longer provides a clean separation between intensity classes.


# Q4. Global Thresholding using Otsu's Method

The assignment specifically asks for an implementation of Otsu's method **from scratch**.

Otsu chooses the threshold \(T\) that maximizes the between-class variance:

\[
\sigma_B^2(T) = \omega_0(T)\omega_1(T)
[\mu_0(T)-\mu_1(T)]^2
\]

where:
- \(\omega_0\), \(\omega_1\) are class probabilities
- \(\mu_0\), \(\mu_1\) are class means

We calculate this for every possible intensity threshold and select the maximum.


In [ ]:
# Select a grayscale image with a reasonably separable histogram
img_q4 = img_as_float(data.camera())

plt.figure(figsize=(9, 6))
plt.imshow(img_q4, cmap="gray")
plt.title("Q4 — Original Grayscale Image")
plt.axis("off")
plt.show()

plt.figure(figsize=(9, 5))
plt.hist(img_q4.ravel(), bins=256, range=(0, 1), density=True)
plt.title("Intensity Histogram")
plt.xlabel("Normalized intensity")
plt.ylabel("Probability density")
plt.show()


### Step 1 — Compute the normalized histogram

For an 8-bit image there are 256 possible intensity levels.

The normalized histogram is:

\[
p(i)=\frac{n_i}{N}
\]

where \(n_i\) is the number of pixels having intensity \(i\), and \(N\) is the total number of pixels.


In [ ]:
def normalized_histogram_256(image):
    image_uint8 = np.clip(image * 255, 0, 255).astype(np.uint8)
    counts = np.bincount(image_uint8.ravel(), minlength=256)
    probabilities = counts / counts.sum()
    return counts, probabilities

counts_q4, probs_q4 = normalized_histogram_256(img_q4)

print("Total pixels:", counts_q4.sum())
print("Probability sum:", probs_q4.sum())
print("Number of intensity levels:", len(probs_q4))

plt.figure(figsize=(10, 5))
plt.bar(np.arange(256), probs_q4, width=1.0)
plt.title("Normalized Histogram / Probability Distribution")
plt.xlabel("Intensity level")
plt.ylabel("Probability")
plt.show()


### Step 2 — Calculate between-class variance for every threshold

For each possible threshold \(T\):
- Class 0 = intensities \(0\) to \(T\)
- Class 1 = intensities \(T+1\) to \(255\)

We calculate their probabilities and means, then evaluate \(\sigma_B^2(T)\).


In [ ]:
def otsu_from_scratch(image):
    counts, probs = normalized_histogram_256(image)
    intensities = np.arange(256, dtype=float)

    between_variance = np.zeros(256, dtype=float)

    # Total mean of the complete image
    total_mean = np.sum(intensities * probs)

    for T in range(256):
        # Class probabilities
        w0 = np.sum(probs[:T+1])
        w1 = np.sum(probs[T+1:])

        # Avoid division by zero
        if w0 == 0 or w1 == 0:
            between_variance[T] = 0
            continue

        # Class means
        mu0 = np.sum(intensities[:T+1] * probs[:T+1]) / w0
        mu1 = np.sum(intensities[T+1:] * probs[T+1:]) / w1

        # Between-class variance
        between_variance[T] = w0 * w1 * (mu0 - mu1) ** 2

    optimal_T = int(np.argmax(between_variance))
    return optimal_T, between_variance

T_otsu, sigma_B = otsu_from_scratch(img_q4)

print("Optimal Otsu threshold (8-bit):", T_otsu)
print("Optimal Otsu threshold (normalized):", T_otsu / 255.0)
print("Maximum between-class variance:", sigma_B[T_otsu])


In [ ]:
# Step 3 — Plot σ_B²(T) versus threshold
plt.figure(figsize=(10, 5))
plt.plot(np.arange(256), sigma_B)
plt.axvline(T_otsu, linestyle="--", linewidth=2,
            label=f"Optimal T = {T_otsu}")
plt.xlabel("Threshold T")
plt.ylabel("Between-class variance σ_B²(T)")
plt.title("Otsu: Between-Class Variance vs Threshold")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


### Step 4 — Apply the optimal threshold

The threshold found above is now applied to the original image:

\[
B(x,y)=
\begin{cases}
0,&I(x,y)\leq T\\
1,&I(x,y)>T
\end{cases}
\]

This produces the final foreground/background segmentation.


In [ ]:
binary_otsu = (img_q4 * 255 > T_otsu)

fig, ax = plt.subplots(1, 2, figsize=(14, 6))

ax[0].imshow(img_q4, cmap="gray")
ax[0].set_title("Original Image")
ax[0].axis("off")

ax[1].imshow(binary_otsu, cmap="gray")
ax[1].set_title(f"Otsu Segmentation — T = {T_otsu}")
ax[1].axis("off")

plt.tight_layout()
plt.show()


### Step 5 — Verify the selected threshold on the histogram

The vertical line below shows where Otsu places the threshold relative to the image intensity distribution.


In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(img_q4.ravel(), bins=256, range=(0, 1), density=True)
plt.axvline(T_otsu / 255.0, linestyle="--", linewidth=2,
            label=f"Otsu T = {T_otsu}")
plt.xlabel("Normalized intensity")
plt.ylabel("Probability density")
plt.title("Histogram with Otsu Threshold")
plt.legend()
plt.show()


## Q4 — Discussion: when does Otsu work well?

### Bimodal histogram
Otsu works particularly well when the histogram contains two reasonably separated groups corresponding to foreground and background. The threshold can then maximize the separation between the two classes.

### Unimodal histogram
If the histogram is approximately unimodal, there may be no clear valley between two classes. Otsu can still return a mathematical optimum, but that threshold may not correspond to a meaningful object/background boundary.

### Significant illumination variation
If illumination changes strongly across the image, pixels belonging to the same physical object can have very different intensities. A single global threshold may then fail. In such cases, adaptive/local thresholding or illumination correction can be more appropriate.

### Important observation
Otsu is a **global** method. It does not explicitly understand objects; it chooses a threshold from the intensity distribution.


# Final Results Summary

Run all cells above first. Then use the following checklist when preparing the report.

| Question | Main implementation | Important outputs |
|---|---|---|
| Q1 | LoG + manual zero crossing + Hough line transform | LoG response, two edge maps, Hough accumulator, line overlays, line counts |
| Q2 | Canny + circular Hough transform | Edge map, radius experiments, threshold experiments, circle overlays |
| Q3 | Gaussian noise + fixed global threshold | Clean/noisy images, three histograms, three binary results, changed-pixel percentages |
| Q4 | **Otsu from scratch** | Histogram, normalized probabilities, σ²B(T) curve, optimal T, final segmentation |

## Key conclusions to report

**Q1:** Increasing the LoG threshold suppresses weak edge responses and can reduce false Hough lines, but an excessively high threshold can remove genuine weak lines.

**Q2:** Circle detection depends strongly on the allowed radius range and accumulator threshold. A good setting balances true detections against false/duplicate circles.

**Q3:** Increasing Gaussian noise broadens the histogram and increases overlap between intensity classes, making fixed global thresholding less reliable.

**Q4:** Otsu automatically selects the threshold that maximizes between-class variance. It is most effective when foreground and background form two reasonably separable intensity classes.

> **Submission tip:** Keep the intermediate figures generated by the notebook in your report. The assignment explicitly asks for intermediate and final outputs, and the parameter comparisons provide evidence for your conclusions.
